<a href="https://colab.research.google.com/github/shashankkumarop-creator/HCL/blob/main/E_Learning_Engagement_Database_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Learning Engagement Database
### Google Colab-ready project

This notebook creates an SQLite database from the supplied e-learning CSV dataset.

- CSV file: `18_elearning_engagement_new_1_to_1000-v2.csv`
- Expected dataset size: 1,000 rows × 12 columns
- SQLite database file: `elearning_engagement.db`
- Main table: `enrollments`


## Step 1 — Upload your CSV file

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import sqlite3
from pathlib import Path

csv_file = next(iter(uploaded))
df = pd.read_csv(csv_file)

print("Dataset shape:", df.shape)
display(df.head())


## Step 2 — Check and prepare the data

In [ ]:
expected_columns = [
    "Enrollment_ID", "Learner_ID", "Course_Name", "Course_Difficulty",
    "Course_Length_Hours", "Hours_Watched", "Quiz_Score", "Quiz_Attempts",
    "Forum_Posts", "Device_Type", "Paid_Tier", "Completed"
]

missing_columns = set(expected_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in CSV: {sorted(missing_columns)}")

df = df[expected_columns].copy()

# IDs are stored as TEXT to preserve values such as EN00001.
df["Enrollment_ID"] = df["Enrollment_ID"].astype(str)
df["Learner_ID"] = df["Learner_ID"].astype(str)

print("Missing values:")
display(df.isnull().sum().to_frame("Missing values"))
print("Duplicate Enrollment_ID values:", df["Enrollment_ID"].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())


## Step 3 — Create the SQLite database

In [ ]:
database_name = "elearning_engagement.db"

conn = sqlite3.connect(database_name)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS enrollments")

cursor.execute('''
CREATE TABLE enrollments (
    Enrollment_ID TEXT PRIMARY KEY,
    Learner_ID TEXT NOT NULL,
    Course_Name TEXT NOT NULL,
    Course_Difficulty TEXT,
    Course_Length_Hours REAL,
    Hours_Watched REAL,
    Quiz_Score REAL,
    Quiz_Attempts INTEGER,
    Forum_Posts INTEGER,
    Device_Type TEXT,
    Paid_Tier TEXT,
    Completed TEXT
)
''')

df.to_sql("enrollments", conn, if_exists="append", index=False)

cursor.execute(
    "CREATE INDEX IF NOT EXISTS idx_enrollments_learner "
    "ON enrollments(Learner_ID)"
)
cursor.execute(
    "CREATE INDEX IF NOT EXISTS idx_enrollments_course "
    "ON enrollments(Course_Name)"
)

conn.commit()
print("SQLite database created:", database_name)


## Step 4 — Verify the database

In [ ]:
# Check the table structure
schema = pd.read_sql_query("PRAGMA table_info(enrollments)", conn)
display(schema)

# Check number of inserted records
record_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM enrollments", conn
)
display(record_count)

# Preview records from the database
display(pd.read_sql_query("SELECT * FROM enrollments LIMIT 10", conn))


## Step 5 — Run SQL analysis queries

In [ ]:
# 1. Course-wise enrollment and completion
course_summary = pd.read_sql_query('''
SELECT
    Course_Name,
    COUNT(*) AS Total_Enrollments,
    SUM(CASE WHEN Completed = 'Yes' THEN 1 ELSE 0 END) AS Completed_Enrollments,
    ROUND(AVG(Quiz_Score), 2) AS Average_Quiz_Score
FROM enrollments
GROUP BY Course_Name
ORDER BY Total_Enrollments DESC
''', conn)
display(course_summary)

# 2. Completion percentage by device
device_summary = pd.read_sql_query('''
SELECT
    Device_Type,
    COUNT(*) AS Total_Enrollments,
    ROUND(
        100.0 * SUM(CASE WHEN Completed = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2
    ) AS Completion_Percent
FROM enrollments
GROUP BY Device_Type
ORDER BY Completion_Percent DESC
''', conn)
display(device_summary)

# 3. Paid vs Free tier
tier_summary = pd.read_sql_query('''
SELECT
    Paid_Tier,
    COUNT(*) AS Total_Enrollments,
    ROUND(AVG(Hours_Watched), 2) AS Average_Hours_Watched,
    ROUND(AVG(Quiz_Score), 2) AS Average_Quiz_Score,
    ROUND(
        100.0 * SUM(CASE WHEN Completed = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2
    ) AS Completion_Percent
FROM enrollments
GROUP BY Paid_Tier
''', conn)
display(tier_summary)

# 4. Learners with the highest total watched hours
learner_activity = pd.read_sql_query('''
SELECT
    Learner_ID,
    ROUND(SUM(Hours_Watched), 2) AS Total_Hours_Watched,
    COUNT(*) AS Total_Enrollments
FROM enrollments
GROUP BY Learner_ID
ORDER BY Total_Hours_Watched DESC
LIMIT 10
''', conn)
display(learner_activity)


## Step 6 — Download the SQLite database

In [ ]:
conn.close()

from google.colab import files
files.download(database_name)


## Database table structure

| Column | SQLite data type | Purpose |
|---|---|---|
| Enrollment_ID | TEXT PRIMARY KEY | Unique enrollment identifier |
| Learner_ID | TEXT | Learner identifier |
| Course_Name | TEXT | Course title |
| Course_Difficulty | TEXT | Course difficulty |
| Course_Length_Hours | REAL | Course length |
| Hours_Watched | REAL | Hours watched |
| Quiz_Score | REAL | Quiz score |
| Quiz_Attempts | INTEGER | Number of quiz attempts |
| Forum_Posts | INTEGER | Number of forum posts |
| Device_Type | TEXT | Device used |
| Paid_Tier | TEXT | Free or Paid tier |
| Completed | TEXT | Completion status |

**Important:** One row represents one course enrollment. A learner can appear in multiple rows, so `Learner_ID` is not the primary key. `Enrollment_ID` uniquely identifies each enrollment.
